In [ ]:
import json
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import clip
from tqdm import tqdm

In [2]:
# =======================================================
# Karpathy-Style Dataset for Flickr30k / COCO
# =======================================================
class KarpathyDataset(Dataset):
    def __init__(self, root, karpathy_json, split="test", transform=None):
        self.root = root
        self.transform = transform

        data = json.load(open(karpathy_json))
        self.entries = []  # list of (img_path, [caption list])

        for item in data["images"]:
            if item["split"] == split:
                img_path = item["filename"]
                captions = [sent["raw"] for sent in item["sentences"]]
                self.entries.append((img_path, captions))

        # flatten captions
        self.all_captions = []
        self.caption2img = []  # caption index -> image index

        for idx, (img_path, caps) in enumerate(self.entries):
            for cap in caps:
                self.all_captions.append(cap)
                self.caption2img.append(idx)

        print(f"[Dataset] {split} split: {len(self.entries)} images, {len(self.all_captions)} captions")

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        img_path, _ = self.entries[idx]
        img = Image.open(os.path.join(self.root, img_path)).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, idx



In [3]:
def recall_i2t(scores_i2t, caption2img, k=1):
    """
    scores_i2t: [N_img, N_cap]
    caption2img: length = N_cap, caption -> GT image id
    """
    N_img = scores_i2t.size(0)
    recalls = 0

    for img_idx in range(N_img):
        # ranking of all captions
        ranking = scores_i2t[img_idx].argsort(descending=True)

        # GT captions for this image
        gt_caps = [i for i in range(len(caption2img)) if caption2img[i] == img_idx]

        # check if any of the GT captions appear in top-k
        if any(cap in ranking[:k] for cap in gt_caps):
            recalls += 1

    return recalls / N_img


def recall_t2i(scores_t2i, caption2img, k=1):
    """
    scores_t2i: [N_cap, N_img]
    caption2img: length = N_cap
    """
    N_cap = len(caption2img)
    recalls = 0

    for cap_idx in range(N_cap):
        ranking = scores_t2i[cap_idx].argsort(descending=True)
        gt_img = caption2img[cap_idx]

        if gt_img in ranking[:k]:
            recalls += 1

    return recalls / N_cap


# =======================================================
# Main Retrieval Evaluation
# =======================================================

def evaluate_retrieval(model, preprocess, dataset, device="cuda", batch_size=64, temp = None):

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # ---------------------
    # Encode Images
    # ---------------------
    print("\nEncoding images...")
    img_feats = []
    with torch.no_grad():
        for imgs, idxs in tqdm(loader):
            imgs = imgs.to(device)
            feat = model.encode_image(imgs)
            img_feats.append(feat)
    img_feats = torch.cat(img_feats, dim=0)  # [N_img, D]

    # ---------------------
    # Encode Captions
    # ---------------------
    print("Encoding captions...")
    cap_feats = []
    B = 256
    caps = dataset.all_captions

    with torch.no_grad():
        for i in tqdm(range(0, len(caps), B)):
            batch_caps = caps[i:i+B]
            txt = clip.tokenize(batch_caps, truncate=True).to(device)
            feat = model.encode_text(txt)
            cap_feats.append(feat)

    cap_feats = torch.cat(cap_feats, dim=0)  # [N_cap, D]

    # ---------------------
    # Similarity Matrices
    # ---------------------
    if temp == None:
        img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
        cap_feats = cap_feats / cap_feats.norm(dim=-1, keepdim=True)
        scores_i2t = img_feats @ cap_feats.T     # [N_img, N_cap]
        scores_t2i = cap_feats @ img_feats.T     # [N_cap, N_img]
    else:
        img_feats = torch.clamp(img_feats, min=0)
        cap_feats = torch.clamp(cap_feats, min=0)
        img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
        cap_feats = cap_feats / cap_feats.norm(dim=-1, keepdim=True)
        scores_i2t = (img_feats @ cap_feats.T) * torch.exp(temp)     # [N_img, N_cap]
        scores_t2i = (cap_feats @ img_feats.T) * torch.exp(temp)    # [N_cap, N_img]


    print("\nSimilarity matrices:")
    print("scores_i2t:", scores_i2t.shape)
    print("scores_t2i:", scores_t2i.shape)
    print("caption2img len:", len(dataset.caption2img))

    # ---------------------
    # Retrieval Metrics
    # ---------------------
    print("\n===== Image → Text Retrieval =====")
    print("R@1 :", recall_i2t(scores_i2t, dataset.caption2img, k=1))
    print("R@5 :", recall_i2t(scores_i2t, dataset.caption2img, k=5))
    print("R@10:", recall_i2t(scores_i2t, dataset.caption2img, k=10))

    print("\n===== Text → Image Retrieval =====")
    print("R@1 :", recall_t2i(scores_t2i, dataset.caption2img, k=1))
    print("R@5 :", recall_t2i(scores_t2i, dataset.caption2img, k=5))
    print("R@10:", recall_t2i(scores_t2i, dataset.caption2img, k=10))


# =======================================================
# Run
# =======================================================
if __name__ == "__main__":
    import clip

    device = "cuda"
    model, preprocess = clip.load("ViT-L/14", device=device, jit=False)

    # -------------------------------------------------------
    # Flickr30k (Karpathy split)
    # -------------------------------------------------------
    # dataset = KarpathyDataset(
    #     root="/MMCBM_datasets/flickr30k/flickr30k-images/",
    #     karpathy_json="/MMCBM_datasets/flickr30k/dataset_flickr30k.json",
    #     split="test",
    #     transform=preprocess
    # )

    # -------------------------------------------------------
    # COCO (Karpathy test split)
    # -------------------------------------------------------
    dataset = KarpathyDataset(
        root="/MMCBM_datasets/coco/val2014/",
        karpathy_json="/MMCBM_datasets/coco/dataset_coco.json",
        split="test",
        transform=preprocess
    )

    evaluate_retrieval(model, preprocess, dataset)

[Dataset] test split: 5000 images, 25010 captions

Encoding images...


100%|██████████| 79/79 [00:15<00:00,  5.03it/s]


Encoding captions...


100%|██████████| 98/98 [00:08<00:00, 11.19it/s]



Similarity matrices:
scores_i2t: torch.Size([5000, 25010])
scores_t2i: torch.Size([25010, 5000])
caption2img len: 25010

===== Image → Text Retrieval =====
R@1 : 0.563
R@5 : 0.7928
R@10: 0.8658

===== Text → Image Retrieval =====
R@1 : 0.3656137544982007
R@5 : 0.6105957616953219
R@10: 0.7114754098360656


In [4]:
import torch
import os
import random
import utils
import data_utils
import json
import clip
from tqdm import tqdm
import random
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from collections import Counter
import cbm_MM
import math
from PIL import Image
import pandas as pd
import csv
import matplotlib.pyplot as plt
import pickle
from visualization import plots

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
load_dir = "/data/saved_models/cc12m_CLIP_ViT-L-14_cbm_2025_10_30_19_22"# path of model weights

with open(os.path.join(load_dir, "args.txt"), "r") as f:
    args = json.load(f)

_, target_preprocess = data_utils.get_target_model(args["img_backbone"], device)
weights = torch.load(os.path.join(load_dir ,"proj_model_weights.pt"), map_location=device)
model = cbm_MM.CBM_model(args["img_backbone"], weights, device=device)
model.eval()

evaluate_retrieval(model, target_preprocess, dataset, temp = model.temp())


Encoding images...


100%|██████████| 79/79 [00:15<00:00,  5.06it/s]


Encoding captions...


100%|██████████| 98/98 [00:08<00:00, 11.40it/s]



Similarity matrices:
scores_i2t: torch.Size([5000, 25010])
scores_t2i: torch.Size([25010, 5000])
caption2img len: 25010

===== Image → Text Retrieval =====
R@1 : 0.4186
R@5 : 0.6782
R@10: 0.7806

===== Text → Image Retrieval =====
R@1 : 0.328828468612555
R@5 : 0.5848460615753699
R@10: 0.6890043982407037
